# 01 · Exploratory Data Analysis & Cleaning

Loads the raw Kaggle books dataset, explores missing values and category distributions, then cleans it into `data/processed/books_cleaned.csv` and `data/processed/tagged_description.txt` (used later for embedding).

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.clean import (
    load_raw_books, flag_missing_and_age, drop_incomplete_rows,
    filter_by_description_length, add_title_and_subtitle,
    add_tagged_description, clean_books, save_cleaned_books,
    save_tagged_description_txt,
)
from src import visualization as viz

## 1. Load raw data

In [ ]:
books = load_raw_books()
books

## 2. Missing values

In [ ]:
viz.plot_missing(books)

`average_rating`, `num_pages`, `rating_count` missing instances are probably from a different underlying dataset.

In [ ]:
books = flag_missing_and_age(books)

### Correlation between missingness and other numeric features

In [ ]:
columns_of_interest = ["num_pages", "age_of_book", "missing_description", "average_rating"]
viz.plot_correlation_heatmap(books, columns_of_interest, method="spearman")

Small number of observations with missing values — replacing them would require web scraping, so the simplest safe option is to drop them.

In [ ]:
books[(books["description"].isna()) |
      (books["num_pages"].isna()) |
      (books["average_rating"].isna()) |
      (books["published_year"].isna())]

### Drop incomplete rows

In [ ]:
book_missing = drop_incomplete_rows(books)
book_missing

## 3. Categories

In [ ]:
book_missing["categories"].value_counts().reset_index().sort_values("count", ascending=False)

In [ ]:
viz.plot_categories_treemap(book_missing["categories"], top_n=100, group_size=10)

In [ ]:
viz.plot_top_categories(book_missing["categories"], top_n=36)

## 4. Description length

In [ ]:
book_missing["words_in_description"] = book_missing["description"].str.split().str.len()
viz.plot_histogram(book_missing["words_in_description"], bin_per_value=True, title="Words in Description", filename="05_words_in_description.png")

Books with fewer than 25 words in their description are too short to be useful for embedding/classification, so they're dropped.

In [ ]:
book_missing_25words = filter_by_description_length(book_missing, min_words=25)
book_missing_25words

## 5. Build `title_and_subtitle` and `tagged_description`

In [ ]:
book_missing_25words = add_title_and_subtitle(book_missing_25words)
book_missing_25words = add_tagged_description(book_missing_25words)
book_missing_25words[["title_and_subtitle", "tagged_description"]].head()

## 6. Save the cleaned dataset

(equivalent to calling `clean_books()` directly on the raw data — shown step by step above for exploration purposes)

In [ ]:
cleaned = clean_books(books=None)  # re-runs the full pipeline from scratch on the raw CSV
save_cleaned_books(cleaned)
save_tagged_description_txt(cleaned)

---
Next notebook: **02_vector_search.ipynb**